# Café Data Analytics Data Cleaning and Transformation

In this notebook, the following steps were performed to clean and transform data. As a result of those actions, the cleaned data was exported to CSV files for future analysis. 

1) [Load Raw Data from BigQuery](#load_data)
2) [Data Exploration and Cleaning](#data_clean)
    * [Remove unsuccessful order transactions](#filter_data)
    * [Extract details of order items into seperate fields from the JSON column](#extract_data)
    * [Drop Irrelevant Columns](#drop_cols)
    * [Impute missing values](#impute_null)
    * [Calculate customer churn threshold](#churn_threshold)
    * [Export the cleaned dataframe to Parquet File](#parquet)
<br><br>
3) [Split the cleaned dataframe based on item categories](#split_data)
    * Hot Drinks / Cold Drinks / Kitchen Data
        * Pivot options into headers
        * Re-calculate option price: *option_price = (item_price / quantity) - unit_price*
        * Impute missing options using the most frequent choices
        * (Cold Drinks data) Append `Flavour` option value to item names for detailed product analysis
        * (Kitchen data) Append `Size` option value to item names for detailed product analysis
        * Export pivoted data to separate CSV files
<br><br>
4) [Union all pivoted Hot Drinks, Cold Drinks, Kitchen Data together and export to a single CSV file](#union_all)

In [ ]:
import pandas as pd
import warnings
import json
import numpy as np
from datetime import datetime, date
import os

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
pd.set_option('max_colwidth', 2000)

<a name="load_data"></a>
## Load Raw Data from BigQuery

In [ ]:
# Load data from BigQuery into local notebook
df_raw = pd.read_gbq(
    """
        SELECT *
        FROM `jr-data-training.cafe.cafe-sales`
    """,
    project_id='jr-data-training',
    location='australia-southeast1',
)

In [ ]:
# Check the first 5 rows of the raw data
df_raw.head()

In [ ]:
# Check the schema of raw data
print(df_raw.shape)
print(df_raw.info())

The `items` column contains JSON strings of order details. We need to extract fields from this column.

<a name="data_clean"></a>
## Data Exploration and Cleaning

In [ ]:
df = df_raw.copy()

<a name="filter_data"></a>
### Remove Unsuccessful Order Transactions

In [ ]:
df_filtered = df[
    ~pd.isna(df['date_paid'])
]

In [ ]:
df_filtered['status'].unique()

Only the status 1 and 2 represent successful transactions, while 10 indicates an unknown status. We'll remove records with status 10 to streamline the dataset.

In [ ]:
df_filtered = df_filtered[
    df_filtered['status'] != 10
].reset_index(drop=True)

In [ ]:
df_filtered['status'].unique()

In [ ]:
print(df_filtered.shape)
print(df_filtered.info())

<a name="extract_data"></a>
### Extract Field Values

In [ ]:
df_extracted = df_filtered.copy()

In [ ]:
# Try converting the JSON string in 'items' column to dict type
try:
    df_extracted['items'] = df_extracted['items'].apply(
        lambda x: json.loads(x)
    )
except Exception as e:
    print(e)

In [ ]:
# View the first items value
df_extracted['items'][0]

In [ ]:
# Extract preliminary fields from the 'items' column
for field_name in df_extracted['items'][0].keys():
    df_extracted[field_name] = df_extracted['items'].str[field_name]

In [ ]:
df_extracted.columns

In [ ]:
# Sample the first extracted 'cart' column 
# that contains lists of ordered items
df_extracted.loc[0, 'cart']

We need to break down the lists in the `cart` column so that each purchased item in an order is placed in its own row.

In [ ]:
df_extracted['cart'].apply(
    lambda x: type(x)
).value_counts()

The `cart` column contains not only list values but also dict values.

In [ ]:
# Sample a 'cart' value that is in dict type
df_extracted['cart'].apply(
    lambda x: x if type(x) == dict else None
).value_counts().index[0]

In [ ]:
def convert_to_list(x):
    if type(x) == dict:
        return list(x.values())
    else:
        return x
    
df_extracted['cart'] = df_extracted['cart'].apply(convert_to_list)

In [ ]:
df_extracted['cart'].apply(lambda x: type(x)).value_counts()

In [ ]:
# Separate elements in the `cart` array into multiple rows
df_extracted = df_extracted.explode('cart', ignore_index=True)
df_extracted[['order_id', 'cart']].head()

In [ ]:
# Extract 'item', 'quantity', 'category', and 'price' fields from the 'cart' column
df_extracted['item'] = df_extracted['cart'].str['name']
df_extracted['quantity'] = df_extracted['cart'].apply(
    lambda x: x['quantity'] if 'quantity' in x else 1
)
df_extracted['category'] = df_extracted['cart'].str['category']
df_extracted['price'] = df_extracted['cart'].str['price']

In [ ]:
# Check if the fields were extracted correctly
df_extracted[['order_id', 'item', 'category', 'quantity', 'price']].head()

In [ ]:
# Print the unique categories of sold items
df_extracted['category'].unique()

In [ ]:
# Extract options details from 'cart' column
df_extracted['options'] = df_extracted['cart'].str['options']

In [ ]:
# Convert options format
def extract_options_list(ops):
    if type(ops) == list:
        options_list = []
        for op in ops:
            options_list.append(
                [
                    op['name'],
                    op['value'],
                    op['price'],
                ]
            )
        return options_list
    else:
        return ops

df_extracted['options'] = df_extracted['options'].apply(extract_options_list)

In [ ]:
df_extracted[
    ['order_id', 'item', 'options', 'quantity', 'price']
].head()

In [ ]:
# Add 'item_tracking_id' to track the number of items in each order
df_extracted['item_tracking_id'] = (
    df_extracted.sort_values(
        ['order_id','item'], ascending=[True, True]
    ).groupby(['order_id']).cumcount() + 1
)

In [ ]:
df_extracted[
    ['order_id', 'item', 'item_tracking_id', 'options']
].head(10)

In [ ]:
# Separate elements in the `options` list into multiple rows
df_extracted = df_extracted.explode('options', ignore_index=True)

In [ ]:
df_extracted[
    ['order_id', 'item', 'options', 'quantity', 'price']
].head()

In [ ]:
# Extract option's name, value and price from 'options' column
df_extracted['option_name'] = df_extracted['options'].apply(
    lambda x: x[0] if type(x) == list else x
)
df_extracted['option_value'] = df_extracted['options'].apply(
    lambda x: x[1] if type(x) == list else x
)
df_extracted['option_price'] = df_extracted['options'].apply(
    lambda x: x[2] if type(x) == list else x
)

In [ ]:
df_extracted[
    ['order_id', 'item', 'options', 'option_name', 'option_value', 'option_price']
].head()

In [ ]:
# Extract 'size' column
df_extracted['size'] = df_extracted.apply(
    lambda x: x['option_value']
    if x['option_name'] == 'size'
    else None,
    axis=1,
)

In [ ]:
# Extract 'unit_price' column
df_extracted['unit_price'] = df_extracted.apply(
    lambda x: x['option_price']
    if x['option_name'] == 'size'
    else None,
    axis=1,
)

In [ ]:
# Create a new dataframe that contains size and unit_price values for each item per order
df_unitprice = (
    df_extracted[
        (~pd.isna(df_extracted['size'])) 
        & (~pd.isna(df_extracted['unit_price']))
      ][
        ['order_id', 'item_tracking_id', 'size', 'unit_price']
    ]
)

df_unitprice.head()

In [ ]:
# Append 'size' and 'unit_price' columns to the master 
# dataframe
df_extracted = df_extracted.drop(
    ['size', 'unit_price'], axis=1
).merge(
    df_unitprice,
    how='left',
    on=['order_id', 'item_tracking_id'],
)

In [ ]:
df_extracted.columns

In [ ]:
df_extracted[
    [
        'order_id', 'item_tracking_id', 'item', 'option_name', 
        'option_value', 'option_price', 'size', 'unit_price'
    ]
].head()

<a name="drop_cols"></a>
### Drop Irrelevant Columns

In [ ]:
df_dropped = df_extracted.copy()

In [ ]:
# Remove the columns containing duplicate or redundant information
df_dropped.drop(
    ['items', 'cart', 'options'], 
    axis=1, 
    inplace=True,
)

In [ ]:
# Check the data type and unique values of each column
col_details = []
for col in df_dropped.columns:
    col_details.append(
        [
            col, 
            df_dropped[col].dtype,
            df_dropped[col].nunique(), 
            df_dropped[col].unique()[:10],
        ],
    )
    
pd.DataFrame(
    col_details,
    columns=[
        'Column Name', 
        'Data Type', 
        'Number of Unique Values', 
        'Unique Value Examples',
    ],
)

In [ ]:
df_dropped[
    ['date_created', 'date_paid', 'order_time', 'order_time_verbose']
].drop_duplicates()[:5]

Let's convert `order_time` column to *datetime* format and substitute `ASAP` values with the corresponding `date_paid` values.

In [ ]:
# Create a temporary 'order_time_' column that converts 'order_time'
# to datetime format and replace 'ASAP' with corresponding 'date_paid'
df_dropped['order_time_'] = df_dropped.apply(
    lambda x: datetime.fromtimestamp(int(x['order_time']))
    if x['order_time'] != 'ASAP' 
    else x['date_paid'],
    axis=1,
)

In [ ]:
df_dropped[
    ['date_created', 'date_paid', 'order_time', 
     'order_time_', 'order_time_verbose']
].drop_duplicates()[:5]

The other time columns including `date_created`, `date_paid`, `order_time`, `order_time_verbose` are redundant and should be dumped.

In [ ]:
# Drop the redundant timeseries columns
df_dropped.drop(
    [
        'date_created',
        'date_paid',
        'order_time',
        'order_time_verbose',
    ],
    axis=1,
    inplace=True,
)

In [ ]:
# Rename 'order_time_' to 'order_time'
df_dropped.rename(
    columns={'order_time_': 'order_time'}, inplace=True
)

In [ ]:
# Investigate the '_display' columns and their corresponding numeric columns
df_dropped[
    ['cart_gst', 'cart_gst_display',
     'cart_surcharge', 'cart_surcharge_display',
     'cart_total_price', 'cart_total_price_display']
].drop_duplicates().head()

The `cart_gst_display`, `cart_surcharge_display`, and `cart_total_price_display` columns represent the string-formatted prices of the corresponding numeric values in the `cart_gst`, `cart_surcharge`, `cart_total_price` columns. Therefore, those `_display` columns should be removed, and the numeric values should be divided by 100 to accurately reflect the true prices.

In [ ]:
df_dropped.drop(
    [
        'cart_surcharge_display', 
        'cart_total_price_display', 
        'cart_gst_display',
    ],
    axis=1, 
    inplace=True,
)

In [ ]:
df_dropped.select_dtypes(include='number').drop_duplicates().head()

Divide the quantitative pricing columns, including `total`, `cart_surcharge`, `cart_total_price`, `cart_gst`, `price`, and `option_price`, by 100 to accurately reflect the true prices.

In [ ]:
cols_to_adjust = [
    'total', 'cart_surcharge', 'cart_total_price', 
    'cart_gst', 'price', 'option_price', 'unit_price'
]

for col in cols_to_adjust:
    df_dropped[col] = df_dropped[col] / 100

In [ ]:
df_dropped[cols_to_adjust].drop_duplicates().head()

Since the `cart_gst` and `cart_surcharge` columns have little relevance to our analysis, and `total` is the same as `cart_total_price`, we can safely remove the three fields from the dataframe to reduce redundancy.

In [ ]:
# Drop 'cart_gst', 'cart_gst' and 'total'
df_dropped.drop(
    [
        'cart_gst',
        'cart_surcharge',
        'total',
    ], 
    axis=1, 
    inplace=True,
)

In [ ]:
# Rename 'cart_total_price' and 'price' appropriately
df_dropped.rename(
    columns={
        'cart_total_price': 'order_price', 
        'price': 'item_price',
        'cart_size': 'order_item_count',
    },
    inplace=True,
)

The fields like `ip_addr`, `order_phone`, `order_name`, and `status` also have little relevance to the analysis and can be removed as well to streamline the dataset.

In [ ]:
df_dropped.drop(
    [
        'ip_addr', 
        'order_phone',
        'order_name',
        'status',
    ], 
    axis=1, 
    inplace=True,
)

In [ ]:
df_dropped.columns

<a name="impute_null"></a>
### Impute NULL values

In [ ]:
df_imputed = df_dropped.copy()

In [ ]:
# Identify the columns with missing values
cols_with_null = []
for col in df_imputed.columns:
    if any(pd.isna(x) for x in df_imputed[col]):
        cols_with_null.append(col)

In [ ]:
cols_with_null

#### Fill in the missing values for the `category` field

In [ ]:
# Print the unique values of category
df_imputed['category'].unique()

In [ ]:
# Find the items that have multiple categories
def find_items_multiple_cat(df):
    df_ = (
        df[['item', 'category']]
        [~pd.isna(df['category'])]
        .drop_duplicates().groupby('item').count()
    )
    return list(df_[df_['category'] > 1].index)

items_multiple_cat = find_items_multiple_cat(df_imputed)
items_multiple_cat

In [ ]:
# Check what categories the target items have been assigned to
df_imputed[
    (df_imputed['item'].isin(items_multiple_cat)) 
    & (~pd.isna(df_imputed['category']))
][['item', 'category']].drop_duplicates().sort_values('item')

It appears that some `Kitchen` or `Food` items have been also been incorrectly classified under `Cold Drinks`, while others have been assigned to both the `Kitchen` and `Food` categories.

In [ ]:
# Create a dictionary for reclassifying the target items
items_recat = {
    'Banana Bread': 'Food',
    'Cookies': 'Food',
    'Croissant': 'Food',
    'Focaccia': 'Food',
    'Fried Chicken on Ciabatta': 'Kitchen',
    'Homemade Sausage Roll': 'Food',
    'Muffin': 'Food',
    'Quiche': 'Food',
    'Toasted Roll': 'Food',
    'Toastie': 'Food',
}

In [ ]:
# Update the target items' category in the master dataframe
for key, value in items_recat.items():
    df_imputed.loc[
        df_imputed[
            df_imputed['item'] == key
        ].index,
        'category'
    ] = value

In [ ]:
# Re-check what categories the target items are now assigned to
df_imputed[
    (df_imputed['item'].isin(items_multiple_cat)) 
    & (~pd.isna(df_imputed['category']))
][['item', 'category']].drop_duplicates().sort_values('item')

In [ ]:
# Check if there still are items that were assigned to 
# multiple categories
find_items_multiple_cat(df_imputed)

Now the categorisation conflicts have successfully been resolved.

In [ ]:
df_imputed['category'].unique()

In [ ]:
# Find the list of items where the 'category' field is missing
items_missing_cat = list(
    sorted(df_imputed[pd.isna(df_imputed['category'])]['item'].unique())
)
items_missing_cat

In [ ]:
# Create an dictionary with items as keys and categories as values
df_item_cat = (
    df_imputed[~pd.isna(df_imputed['category'])]
    [['item', 'category']].drop_duplicates().sort_values('item')
)
dict_item_cat = {x[0]: x[1] for x in df_item_cat.to_numpy()}
dict_item_cat

In [ ]:
# Find the items that are still not classified
items_not_classified = [
    item for item in items_missing_cat if item not in dict_item_cat
]
items_not_classified

In [ ]:
# View the option_name and option_value associated with the unclassified items
df_imputed[df_imputed['item'].isin(items_not_classified)][
    ['item', 'option_name', 'option_value']
].sort_values('item')

Since `Toasties` and `Toastie` refer to the same item, they should both be classified under `Kitchen`. Given that the `Short Black` item shares similar options with other `Hot Drinks` items, it should be classified under `Hot Drinks`. Additionally, `Spider`, based on its options, aligns more with the `Cold Drinks` category.

In [ ]:
df_imputed['item'] = df_imputed['item'].apply(
    lambda x: 'Toastie' if x == 'Toasties' else x
)

In [ ]:
# Reflect the re-categorisation of 'Short Black' and 
# 'Spider' in dict_item_cat
dict_item_cat['Short Black'] = 'Hot Drinks'
dict_item_cat['Spider'] = 'Cold Drinks'

In [ ]:
def impute_category(x):
    if pd.isna(x['category']):
        return dict_item_cat[x['item']]
    else:
        return x['category']
    
df_imputed['category'] = df_imputed.apply(impute_category, axis=1)

In [ ]:
df_imputed['category'].unique()

Now all items have been categorised.

Before we move on to the next step, let's review all the items that have been classified under `Hot Drinks`, `Cold Drinks`, `Food` and `Kitchen`.

### Reclassify Items Appropriately

In [ ]:
def view_classified_items(df):
    return pd.DataFrame(
        {
            'category': df['category'].unique(),
            'item': [
                df[df['category'] == cat]['item'].unique() 
                for cat in df['category'].unique()
            ],
        }
    )

In [ ]:
# Review all classified items
view_classified_items(df_imputed)

It appears that some `Food` and `Kitchen` items have been incorrectly assigned to the `Cold Drinks` category. Let's correct these misclassifications.

In [ ]:
items_recat = {
    'Brownie': 'Food',
    'Scone': 'Food',
    'Friand (GF)': 'Food',
    'Carrot Cake': 'Food',
    'Orange Almond Cake (VG)': 'Food',
    'Orange Almond Cake (GF,VG)': 'Food',
    'Chips and Wedges': 'Food',
    'Fried Chicken Sandwich': 'Kitchen',
}

In [ ]:
# Correct the misclassified items' category in the master dataframe
for key, value in items_recat.items():
    df_imputed.loc[
        df_imputed[
            df_imputed['item'] == key
        ].index,
        'category'
    ] = value

In [ ]:
view_classified_items(df_imputed)

Now all items seem appropriately categorised. Let's move on to calculate customer churn threshold for each customer for future customer churn analysis.

<a name="churn_threshold"></a>
### Calculate Customer Churn Threshold

In [ ]:
df_final = df_imputed.copy()

In [ ]:
# Create a new dataframe for labeling churns
df_churn = df_final[
    ['customer_id', 'order_time', 'order_id']
].drop_duplicates().sort_values(
    ['customer_id', 'order_time', 'order_id'],
    ascending=[True, True, True],
).reset_index(drop=True).copy()

df_churn.head()

In [ ]:
# Create a new column 'next_order_time' 
# for calculating purchase intervals
df_churn['next_order_time'] = df_churn.sort_values(
    by=['customer_id', 'order_time'], 
    ascending=[True, True],
).groupby(['customer_id'])['order_time'].shift(-1)

df_churn.head()

In [ ]:
# Create new column for recording purchasing interval in days
def get_interval(x):
    if pd.isna(x['next_order_time']):
        return None
    else:
        return (
            x['next_order_time'].date()
            - x['order_time'].date()
        ).days

df_churn['purchase_interval_days'] = (
    df_churn.apply(
        lambda x: get_interval(x), axis=1
    )
)

df_churn.head()

Compute the **mean** and **standard deviation** of purchase interval days for each customer, and set the churn threshold at **one standard deviation above the mean purchase interval**.

In [ ]:
# Calculate the mean of purchasing interval days
# for each customer
df_churn['interval_mean'] = (
    df_churn.groupby('customer_id')
    ['purchase_interval_days'].transform(np.mean)
)

# Calculate the standard deviation of purchasing 
# interval days for each customer
df_churn['interval_std'] = (
    df_churn.groupby('customer_id')
    ['purchase_interval_days'].transform(np.std)
)

# Calculate the churn threshold for each customer as 
# one standard deviation above the interval mean
df_churn['churn_threshold'] = (
    df_churn['interval_mean'] + 1 * df_churn['interval_std']
)

In [ ]:
df_churn.head(10)

In [ ]:
# Create a new dataframe that maps the churn thresholds 
# to each customer
df_customer_churn = df_churn[
    ['customer_id', 'churn_threshold']
].drop_duplicates().reset_index(drop=True)

print(df_customer_churn.shape)
print(df_customer_churn.info())
df_customer_churn.head(10)

For customers with NaN values in the `churn_threshold`, it likely indicates they have only placed one or two orders. In such cases, their churn threshold can be set as one standard deviation above the mean purchase interval calculated **across the entire customer base**.

In [ ]:
# Compute one standard deviation above the average purchase 
# interval calculated across the entire customer base
churn_threshold_all = (
    np.mean(df_churn['purchase_interval_days'])
    + 1 * np.std(df_churn['purchase_interval_days'])
)
print(churn_threshold_all)

In [ ]:
# Impute NaN in 'churn_threshold'
df_customer_churn['churn_threshold'] = (
    df_customer_churn['churn_threshold'].apply(
        lambda x: churn_threshold_all if pd.isna(x) else x
    )
)

df_customer_churn.head(10)

In [ ]:
# Append the churn threshold to the master dataframe
df_final = df_final.merge(
    df_customer_churn,
    how='left',
    on='customer_id',
)

In [ ]:
df_final.columns

In [ ]:
df_final.head()

In [ ]:
print(df_final.shape)
print(df_final.info())

<a name="parquet"></a>
### Export to Parquet File

In [ ]:
# Create the output folder if it doesn't exist
output_folder = 'cafe_processed_files'
if not os.path.exists(os.path.join(os.getcwd(), output_folder)):
    os.mkdir(output_folder)

# Save the final cleaned data to a PARQUET file
df_final.to_parquet(
    os.path.join(output_folder, 'cafe_full_mx.parquet'),
)

<hr>

<a name="split_data"></a>
## Pivot Option to Columns

In [ ]:
df_unpivoted = df_final.copy()

In [ ]:
# Display the option names and item sizes for each category
# in a dataframe
def display_option_size(df):
    option_names = [
        df[
            df['category'] == cat
        ]['option_name'].unique()
        for cat in df['category'].unique()
    ]

    sizes = [
        df[
            df['category'] == cat
        ]['size'].unique()
        for cat in df['category'].unique()
    ]

    return pd.DataFrame(
        {
            'Item Category': df['category'].unique(),
            'Option Names': option_names,
            'Size': sizes,
        }
    )

display_option_size(df_unpivoted)

* The `Size` values other than *MINI*, *SML*, *REG*, *LRG*, *XLG* can be appended to the respective item names to facilitate future product analysis.
* The `Extra` and `Extras` options for **Kitchen** seem identical, so we should rename them to a consistent name to prevent confusion.
* Similarly, the `Flavour` option values for **Cold Drinks** items can also be appended to item names to further support product analysis.

In [ ]:
extra_item_count = df_unpivoted[
    df_unpivoted['option_name'] == 'Extra'
]['item'].nunique()

extras_item_count = df_unpivoted[
    df_unpivoted['option_name'] == 'Extras'
]['item'].nunique()

print(f'{extra_item_count} items have Extra option')
print(f'{extras_item_count} items have Extras option')

In [ ]:
# Rename 'Extra' to 'Extras' option
df_unpivoted['option_name'] = df_unpivoted['option_name'].apply(
    lambda x: 'Extras' if x == 'Extra' else x
)

In [ ]:
# Append 'Size' values to corresponding item names
df_unpivoted['variation'] = df_unpivoted.apply(
    lambda x: x['item'] + '_' + x['size']
    if x['size'] not in ['MINI', 'SML', 'REG', 'LRG', 'XLG']
    else x['item'],
    axis=1,
)

df_unpivoted['size'] = df_unpivoted['size'].apply(
    lambda x: None if x not in ['MINI', 'SML', 'REG', 'LRG', 'XLG']
    else x
)

In [ ]:
df_unpivoted['option_value'] = df_unpivoted.apply(
    lambda x: None if x['option_name'] == 'size'
    else x['option_value'],
    axis=1,
)

df_unpivoted['option_name'] = (
    df_unpivoted['option_name'].apply(
        lambda x: None if x == 'size' else x
    )
)

In [ ]:
# Sample order 29630 to see the transformation effect 
df_unpivoted[
    df_unpivoted['order_id'] == 29630
][
    [
        'order_id', 'customer_id', 'item_tracking_id', 
        'item', 'variation', 'option_name', 'option_value', 
        'size'
    ]
].drop_duplicates().sort_values('item_tracking_id', ascending=True)

In [ ]:
df_unpivoted['variation'].nunique()

In [ ]:
df_unpivoted['item'].nunique()

In [ ]:
df_unpivoted['category'].nunique()

The DataFrame now includes **4 categories**, **86 unique items**, and **194 variations**.

### Split the DataFrame based on Item Category

In [ ]:
# Split the dataframe based on the 3 categories of sold items
df_hotdrinks = df_unpivoted[df_unpivoted['category'] == 'Hot Drinks'].copy()
df_colddrinks = df_unpivoted[df_unpivoted['category'] == 'Cold Drinks'].copy()
df_kitchen = df_unpivoted[df_unpivoted['category'] == 'Kitchen'].copy()
df_food = df_unpivoted[df_unpivoted['category'] == 'Food'].copy()

### Transform Hot Drinks Data for Better Usability

In [ ]:
df_hotdrinks.head()

#### Pivot `option_name` into headers

In [ ]:
def pivot_options(df, pivot_value_col):
    # Pivot the 'option_name' column into headers, 
    # using the 'option_value' column for their corresponding values
    df_pivoted = df.pivot(
        index=['order_id', 'item_tracking_id'], 
        columns='option_name', 
        values=pivot_value_col,
    ).reset_index(drop=False)
    
    # Gather other attributes for each item per order
    df_attr = df.drop(
        ['option_name', pivot_value_col, 'option_price'], 
        axis=1,
    ).drop_duplicates()

    # Join the two tables to create a final table,
    # ensuring each row represents one item per order
    df_pivoted = df_attr.merge(
        df_pivoted, how='inner', on=['order_id', 'item_tracking_id']
    )
    return df_pivoted

In [ ]:
df_hotdrinks_pivoted = pivot_options(df_hotdrinks, 'option_value')

In [ ]:
df_hotdrinks_pivoted.drop(np.nan, axis=1, inplace=True)

In [ ]:
# Check the pivoted table schema
print(df_hotdrinks_pivoted.shape)
print(df_hotdrinks_pivoted.columns)
df_hotdrinks_pivoted.head()

#### Create a new `option_price` column

In [ ]:
def calc_option_price(df):
    # Create a new 'option_price' column that represents 
    # the difference between 'item_price' per unit and 'unit_price'
    df['option_price'] = (
        df['item_price'] / df['quantity'] - df['unit_price']
    )
    return df

In [ ]:
df_hotdrinks_pivoted = calc_option_price(df_hotdrinks_pivoted)

In [ ]:
# Find the order that includes the highest number of hot drinks
df_hotdrinks_pivoted['order_id'].value_counts()[:1]

In [ ]:
# Check the 'order_price', 'item_price', 'unit_price' 
# and 'option_price' for the sampled order 17135
df_hotdrinks_pivoted[df_hotdrinks_pivoted['order_id'] == 17135][
    [
        'order_id', 'item_tracking_id', 'item', 'size', 'quantity',
        'order_price', 'item_price', 'unit_price', 'option_price'
    ]
].sort_values('item_tracking_id', ascending=True)

Now, let's clean the option columns, including `Decaf`, `Equal Sugar`, `Extra shot`, `Honey`, `Milk`, `Raw Sugar`, `Strength`, `Syrup`, `Temp`, and `White Sugar`.

#### Impute missing options

In [ ]:
option_columns = df_hotdrinks['option_name'].unique()[1:]
option_columns

In [ ]:
def get_item_option_dict(df):
    items = list(df['item'].unique())

    # Create a dictionary with 'item' as keys and their
    # corresponding 'option_name' as values 
    item_option_dict = {}
    for item in items:
        item_option_dict[item] = (
            df[
                df['item'] == item
            ]['option_name'].unique()
            [1:]
        )
        
    return item_option_dict

In [ ]:
hotdrink_item_option_dict = get_item_option_dict(df_hotdrinks)

In [ ]:
# Check the unique options associated with each item
def show_options_per_item(df, item_option_dict, item_cat):    
    # Check if any applicable option column for each item has NULl values
    items = item_option_dict.keys()
    has_null_list = []
    for item in items:
        has_null = np.any(
            pd.isna(df[item_option_dict[item]][
                df['item'] == item
            ]).to_numpy() == True
        )
        has_null_list.append(has_null)
   
    return pd.DataFrame(
        {
            item_cat: item_option_dict.keys(), 
            'Applicable Options': item_option_dict.values(),
            'Has NULL': has_null_list,
        }
    )

In [ ]:
show_options_per_item(
    df_hotdrinks_pivoted, hotdrink_item_option_dict, 'Hot Drinks'
)

In [ ]:
# Impute the applicable option columns only for each item with the most common choice
def impute_options(df, item_option_dict, option_columns):
    has_null_list = []
    for item in item_option_dict.keys():
        # Find the list of applicable option columns for the current item
        item_options = item_option_dict[item]
        for option in option_columns:
            # If the current option column is applicable for the current item
            if option in item_options:
                # Get the most frequent choice for the option
                most_common = df[
                    df['item'] == item
                ][option].value_counts().index[0]

                # Impute the NaN in the option column with the most frequent choice
                df.loc[
                    df[
                        (df['item'] == item)
                        & (pd.isna(df[option]))
                    ].index, option
                ] = most_common

In [ ]:
impute_options(
    df_hotdrinks_pivoted, hotdrink_item_option_dict, option_columns
)

In [ ]:
show_options_per_item(
    df_hotdrinks_pivoted, hotdrink_item_option_dict, 'Hot Drinks'
)

All missing values in the option columns relevant to each item have now been appropriately filled

In [ ]:
def show_unique_options(df, option_columns):    
    unique_options = []
    for col in option_columns:
        unique_options.append(
            list(df[col].unique())
        )

    return pd.DataFrame(
        {
            'Options': option_columns,
            'Unique Values': unique_options,
        }
    )

In [ ]:
show_unique_options(df_hotdrinks_pivoted, option_columns)

In [ ]:
print(df_hotdrinks_pivoted.shape)
print(df_hotdrinks_pivoted.info())

#### Export the Hot Drinks data as a separate Parquet file

In [ ]:
# Create the output folder if it doesn't exist
output_folder = 'cafe_processed_files'
if not os.path.exists(os.path.join(os.getcwd(), output_folder)):
    os.mkdir(output_folder)

# Save the final cleaned data to a PARQUET file
df_hotdrinks_pivoted.to_parquet(
    os.path.join(output_folder, 'cafe_hotdrinks_pivoted_mx.parquet'),
)

### Transform Cold Drinks Data for Better Usability

In [ ]:
df_colddrinks = df_unpivoted[df_unpivoted['category'] == 'Cold Drinks'].copy()

In [ ]:
df_colddrinks.reset_index(drop=True, inplace=True)
print(df_colddrinks.shape)
print(df_colddrinks.info())

In [ ]:
df_colddrinks.head()

In [ ]:
def find_options_with_multiple_values(df):
    # Group the number of option_value by 'order_id',
    # 'item_tracking_id', and 'option_name'
    df_ = df[
        ['order_id', 'item_tracking_id', 'option_name', 'option_value']
    ].groupby(
        ['order_id', 'item_tracking_id', 'option_name']
    ).count().sort_values(
        'option_value', ascending=False
    ).reset_index(drop=False).rename(
        columns={'option_value': 'option_value_count'}
    )
    
    # Find which option(s) has more than one corresponding 'option_value'
    return df_[
        df_['option_value_count'] > 1
    ]['option_name'].unique()

In [ ]:
find_options_with_multiple_values(df_colddrinks)

The `Ingredients` option has more than one corresponding `option_value`. Let's dive in to learn this option...

In [ ]:
df_colddrinks[df_colddrinks['order_id'] == 11329][
    ['order_id', 'item_tracking_id', 'option_name', 'option_value']
]

It is clear to see that some items in each order can have more than one `Ingredient`. We can concatenate these option values into a single string.

#### Concatenate `Ingredient` values for each item into single row

In [ ]:
def concat_options(df):
    # Create a new column 'option_value_new' that concatenates 
    # options for each item per order into a single row
    df['option_value_new'] = df[
        ['order_id', 'item_tracking_id', 'option_name', 'option_value']
    ].groupby(
        ['order_id', 'item_tracking_id', 'option_name']
    ).transform(lambda x: ', '.join(x))

    df = df.drop_duplicates(
        ['order_id', 'item_tracking_id', 'option_name', 'option_value_new']
    )
    return df

In [ ]:
df_colddrinks = concat_options(df_colddrinks)

In [ ]:
# Check if the ingredients have successfully been concatenated
df_colddrinks[df_colddrinks['order_id'] == 11329][
    ['order_id', 'item_tracking_id', 'option_name', 
     'option_value', 'option_value_new']
]

In [ ]:
# Drop the useless 'option_value' column
df_colddrinks.drop('option_value', axis=1, inplace=True)

#### Pivot `option_name` into headers

In [ ]:
df_colddrinks_pivoted = pivot_options(df_colddrinks, 'option_value_new')

In [ ]:
df_colddrinks_pivoted.drop(np.nan, axis=1, inplace=True)

In [ ]:
# Check the pivoted table schema
print(df_colddrinks_pivoted.shape)
print(df_colddrinks_pivoted.columns)
df_colddrinks_pivoted.head()

#### Create a new `option_price` column

In [ ]:
# Create a new 'option_price' column that represents 
# the difference between 'item_price' per unit and 'unit_price'
df_colddrinks_pivoted = calc_option_price(df_colddrinks_pivoted)

In [ ]:
# Find the order that includes the highest number of cold drinks
df_colddrinks_pivoted['order_id'].value_counts()[:1]

In [ ]:
# Check the 'order_price', 'item_price', 'unit_price' 
# and 'option_price' for the sampled order 15001
df_colddrinks_pivoted[df_colddrinks_pivoted['order_id'] == 15001][
    [
        'order_id', 'item_tracking_id', 'item', 'size', 'quantity',
        'order_price', 'item_price', 'unit_price', 'option_price'
    ]
].sort_values('item_tracking_id', ascending=True)

Now, let's clean all the option columns.

#### Impute missing options

In [ ]:
option_columns = df_colddrinks['option_name'].unique()[1:]
option_columns

In [ ]:
# Create a dictionary with 'item' as keys and their
# corresponding 'option_name' as values
colddrink_item_option_dict = get_item_option_dict(df_colddrinks)

In [ ]:
# Check the unique options associated with each item
show_options_per_item(
    df_colddrinks_pivoted, colddrink_item_option_dict, 'Cold Drinks'
)

In [ ]:
impute_options(
    df_colddrinks_pivoted, colddrink_item_option_dict, option_columns
)

In [ ]:
# Check the unique options associated with each item
show_options_per_item(
    df_colddrinks_pivoted, colddrink_item_option_dict, 'Cold Drinks'
)

All missing values in the option columns relevant to each item have now been appropriately filled

In [ ]:
show_unique_options(df_colddrinks_pivoted, option_columns)

#### Append `Flavour` values to item names

In [ ]:
# Append 'Flavour' option values to Cold Drinks item names
df_colddrinks_pivoted['variation'] = df_colddrinks_pivoted.apply(
    lambda x: x['variation'] + '_' + x['Flavour']
    if not pd.isna(x['Flavour'])
    else x['variation'],
    axis=1,
)

In [ ]:
# Drop the 'Flavour' column
df_colddrinks_pivoted.drop('Flavour', axis=1, inplace=True)

In [ ]:
# Check the unique cold drinks item variations
sorted(df_colddrinks_pivoted['variation'].unique())

In [ ]:
df_colddrinks_pivoted.columns

In [ ]:
print(df_colddrinks_pivoted.shape)
print(df_colddrinks_pivoted.info())

#### Export the Cold Drinks data as a separate Parquet file

In [ ]:
# Create the output folder if it doesn't exist
output_folder = 'cafe_processed_files'
if not os.path.exists(os.path.join(os.getcwd(), output_folder)):
    os.mkdir(output_folder)

# Save the final cleaned data to a PARQUET file
df_colddrinks_pivoted.to_parquet(
    os.path.join(output_folder, 'cafe_colddrinks_pivoted_mx.parquet'),
)

### Transform Kitchen Data for Better Usability

In [ ]:
df_colddrinks = df_unpivoted[df_unpivoted['category'] == 'Kitchen'].copy()

In [ ]:
df_kitchen.reset_index(drop=True, inplace=True)
print(df_kitchen.shape)
print(df_kitchen.info())

In [ ]:
# Check the unique options for Kitchen items
option_columns = df_kitchen['option_name'].unique()[1:]
option_columns

In [ ]:
# identify options in the kitchen dataset that have 
# multiple corresponding values for a single item
find_options_with_multiple_values(df_kitchen)

The `Extras`, `Toppings`, and `Options` option may have multiple corresponding `option_value` for a single item. Let's concatenate these values into a single row.

#### Concatenate `Extras`, `Toppings` and `Options` values for each item into single row

In [ ]:
# Create a new column 'option_value_new' that concatenates ingredients
# together into a single row
df_kitchen = concat_options(df_kitchen)

In [ ]:
# Check if the `Extras`, `Toppings`, and `Options` have 
# successfully been concatenated
df_kitchen[df_kitchen['order_id'] == 15178][
    ['order_id', 'item_tracking_id', 'option_name', 
     'option_value', 'option_value_new']
].sort_values(['order_id', 'item_tracking_id'], ascending=[True, True])

In [ ]:
# Drop the useless 'option_value' column
df_kitchen.drop('option_value', axis=1, inplace=True)

#### Pivot `option_name` into headers

In [ ]:
df_kitchen_pivoted = pivot_options(df_kitchen, 'option_value_new')

In [ ]:
df_kitchen_pivoted.drop(np.nan, axis=1, inplace=True)

In [ ]:
# Check the pivoted table schema
print(df_kitchen_pivoted.shape)
print(df_kitchen_pivoted.columns)
df_kitchen_pivoted.head()

#### Create a new `option_price` column

In [ ]:
# Create a new 'option_price' column that represents 
# the difference between 'item_price' per unit and 'unit_price'
df_kitchen_pivoted = calc_option_price(df_kitchen_pivoted)

In [ ]:
# Find the order that includes the highest number of kitchen items
df_kitchen_pivoted['order_id'].value_counts()[:1]

In [ ]:
# Check the 'order_price', 'item_price', 'unit_price' 
# and 'option_price' for the sampled order 15756
df_kitchen_pivoted[df_kitchen_pivoted['order_id'] == 15756][
    [
        'order_id', 'item_tracking_id', 'item', 'size', 'quantity',
        'order_price', 'item_price', 'unit_price', 'option_price'
    ]
]

Now, let's clean all the option columns.

#### Impute missing options

In [ ]:
option_columns = df_kitchen['option_name'].unique()[1:]
option_columns

In [ ]:
# Create a dictionary with 'item' as keys and their
# corresponding 'option_name' as values
kitchen_item_option_dict = get_item_option_dict(df_kitchen)

In [ ]:
# Check the unique options associated with each item
show_options_per_item(
    df_kitchen_pivoted, kitchen_item_option_dict, 'Kitchen Items'
)

In [ ]:
impute_options(
    df_kitchen_pivoted, kitchen_item_option_dict, option_columns
)

In [ ]:
# Check the unique options associated with each item
show_options_per_item(
    df_kitchen_pivoted, kitchen_item_option_dict, 'Kitchen Items'
)

All missing values in the option columns relevant to each item have now been appropriately filled.

In [ ]:
show_unique_options(df_kitchen_pivoted, option_columns)

In [ ]:
print(df_kitchen_pivoted.shape)
print(df_kitchen_pivoted.info())

#### Export the Kitchen data as a separate Parquet file

In [ ]:
# Create the output folder if it doesn't exist
output_folder = 'cafe_processed_files'
if not os.path.exists(os.path.join(os.getcwd(), output_folder)):
    os.mkdir(output_folder)

# Save the final cleaned data to a PARQUET file
df_kitchen_pivoted.to_parquet(
    os.path.join(output_folder, 'cafe_kitchen_pivoted_mx.parquet'),
)

### Transform Food Data for Better Usability

In [ ]:
df_food = df_unpivoted[df_unpivoted['category'] == 'Food'].copy()

In [ ]:
df_food.reset_index(drop=True, inplace=True)
print(df_food.shape)
print(df_food.info())

In [ ]:
# Check the unique options for Food items
option_columns = df_food['option_name'].unique()[1:]
option_columns

In [ ]:
# identify options in the Food dataset that have 
# multiple corresponding values for a single item
find_options_with_multiple_values(df_food)

The `Extras` option may have multiple corresponding `option_value` for a single item. Let's concatenate these values into a single row.

#### Concatenate `Extras` values for each item into single row

In [ ]:
# Create a new column 'option_value_new' that concatenates ingredients
# together into a single row
df_food = concat_options(df_food)

In [ ]:
# Check if the `Extras` have successfully been concatenated
df_food[df_food['order_id'] == 19889][
    ['order_id', 'item_tracking_id', 'option_name', 
     'option_value', 'option_value_new']
].sort_values(['order_id', 'item_tracking_id'], ascending=[True, True])

In [ ]:
# Drop the useless 'option_value' column
df_food.drop('option_value', axis=1, inplace=True)

#### Pivot `option_name` into headers

In [ ]:
df_food_pivoted = pivot_options(df_food, 'option_value_new')

In [ ]:
df_food_pivoted.drop(np.nan, axis=1, inplace=True)

In [ ]:
# Check the pivoted table schema
print(df_food_pivoted.shape)
print(df_food_pivoted.columns)
df_food_pivoted.head()

#### Create a new `option_price` column

In [ ]:
# Create a new 'option_price' column that represents 
# the difference between 'item_price' per unit and 'unit_price'
df_food_pivoted = calc_option_price(df_food_pivoted)

In [ ]:
# Find the order that includes the highest number of kitchen items
df_food_pivoted['order_id'].value_counts()[:1]

In [ ]:
# Check the 'order_price', 'item_price', 'unit_price' 
# and 'option_price' for the sampled order 25393
df_food_pivoted[df_food_pivoted['order_id'] == 25393][
    [
        'order_id', 'item_tracking_id', 'item', 'size', 'quantity',
        'order_price', 'item_price', 'unit_price', 'option_price'
    ]
]

Now, let's clean all the option columns.

#### Impute missing options

In [ ]:
option_columns = df_food['option_name'].unique()[1:]
option_columns

In [ ]:
# Create a dictionary with 'item' as keys and their
# corresponding 'option_name' as values
food_item_option_dict = get_item_option_dict(df_food)

In [ ]:
# Check the unique options associated with each item
show_options_per_item(
    df_food_pivoted, food_item_option_dict, 'Food Items'
)

In [ ]:
impute_options(
    df_food_pivoted, food_item_option_dict, option_columns
)

In [ ]:
# Check the unique options associated with each item
show_options_per_item(
    df_food_pivoted, food_item_option_dict, 'Food Items'
)

All missing values in the option columns relevant to each item have now been appropriately filled.

In [ ]:
show_unique_options(df_food_pivoted, option_columns)

In [ ]:
print(df_food_pivoted.shape)
print(df_food_pivoted.info())

#### Export the Food data as a separate Parquet file

In [ ]:
# Create the output folder if it doesn't exist
output_folder = 'cafe_processed_files'
if not os.path.exists(os.path.join(os.getcwd(), output_folder)):
    os.mkdir(output_folder)

# Save the final cleaned data to a PARQUET file
df_food_pivoted.to_parquet(
    os.path.join(output_folder, 'cafe_food_pivoted_mx.parquet'),
)

<a name="union_all"></a>
### Union all Pivoted Dataframes into a Single One and Export to Parquet File

In [ ]:
df_pivoted = pd.concat(
    [
        df_hotdrinks_pivoted,
        df_colddrinks_pivoted,
        df_kitchen_pivoted,
    ],
    axis=0,
    ignore_index=True,
)

In [ ]:
# Check the unique number of products
df_pivoted['item'].nunique()

In [ ]:
print(df_pivoted.shape)
print(df_pivoted.info())

In [ ]:
# Create the output folder if it doesn't exist
output_folder = 'cafe_processed_files'
if not os.path.exists(os.path.join(os.getcwd(), output_folder)):
    os.mkdir(output_folder)

# Save the final cleaned data to a PARQUET file
df_pivoted.to_parquet(
    os.path.join(output_folder, 'cafe_full_pivoted_mx.parquet'),
)

<hr>